In [8]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.preprocessing import LabelEncoder

# Set the style for better visualization
plt.style.use('ggplot')
sns.set(font_scale=1.2)

# Load the data
df = pd.read_csv('students.csv')

In [9]:
# Display basic information
print("Dataset Shape:", df.shape)
print("\nFirst 5 rows:")
print(df.head())

Dataset Shape: (395, 6)

First 5 rows:
  school address  absences     Mjob      Fjob  G3
0     GP       U         6  at_home   teacher   6
1     GP       U         4  at_home     other   6
2     GP       U        10  at_home     other  10
3     GP       U         2   health  services  15
4     GP       U         4    other     other  10


In [10]:
print("\nData Information:")
print(df.info())

print("\nSummary Statistics:")
print(df.describe())

print("\nMissing Values:")
print(df.isnull().sum())


Data Information:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 395 entries, 0 to 394
Data columns (total 6 columns):
 #   Column    Non-Null Count  Dtype 
---  ------    --------------  ----- 
 0   school    395 non-null    object
 1   address   395 non-null    object
 2   absences  395 non-null    int64 
 3   Mjob      395 non-null    object
 4   Fjob      395 non-null    object
 5   G3        395 non-null    int64 
dtypes: int64(2), object(4)
memory usage: 18.6+ KB
None

Summary Statistics:
         absences          G3
count  395.000000  395.000000
mean     5.708861   10.415190
std      8.003096    4.581443
min      0.000000    0.000000
25%      0.000000    8.000000
50%      4.000000   11.000000
75%      8.000000   14.000000
max     75.000000   20.000000

Missing Values:
school      0
address     0
absences    0
Mjob        0
Fjob        0
G3          0
dtype: int64


In [11]:
# Check unique values in categorical columns
print("\nUnique Values in Categorical Columns:")
for col in df.select_dtypes(include=['object']).columns:
    print(f"{col}: {df[col].unique()}")


Unique Values in Categorical Columns:
school: ['GP' 'MS']
address: ['U' 'R']
Mjob: ['at_home' 'health' 'other' 'services' 'teacher']
Fjob: ['teacher' 'other' 'services' 'health' 'at_home']


In [12]:
df['is_urban'] = df['address'].map({'U': 1, 'R': 0})

df['parents_same_field'] = (df['Mjob'] == df['Fjob']).astype(int)

bins = [-1, 0, 5, 10, 20, 100]
labels = ['None', 'Low', 'Medium', 'High', 'Very High']
df['absence_category'] = pd.cut(df['absences'], bins=bins, labels=labels)

df['performance_category'] = pd.cut(df['G3'],
                                   bins=[-1, 7, 12, 15, 20],
                                   labels=['Poor', 'Average', 'Good', 'Excellent'])

job_categories = {
    'health': 'Healthcare',
    'services': 'Services',
    'teacher': 'Education',
    'at_home': 'Home',
    'other': 'Other'
}

df['mother_job_category'] = df['Mjob'].map(job_categories)
df['father_job_category'] = df['Fjob'].map(job_categories)

In [13]:
df_mother_job = pd.get_dummies(df['mother_job_category'], prefix='mother')
df_father_job = pd.get_dummies(df['father_job_category'], prefix='father')
df = pd.concat([df, df_mother_job, df_father_job], axis=1)

In [14]:
plt.figure(figsize=(10, 6))
sns.histplot(df['G3'], kde=True, bins=20)
plt.title('Distribution of Final Grades (G3)')
plt.xlabel('Final Grade')
plt.ylabel('Count')
plt.savefig('grade_distribution.png')
plt.close()

In [15]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='school', y='G3', data=df)
plt.title('Final Grade Distribution by School')
plt.xlabel('School')
plt.ylabel('Final Grade (G3)')
plt.savefig('grade_by_school.png')
plt.close()

In [16]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='address', y='G3', data=df)
plt.title('Final Grade Distribution by Address Type')
plt.xlabel('Address Type (U=Urban, R=Rural)')
plt.ylabel('Final Grade (G3)')
plt.savefig('grade_by_address.png')
plt.close()

In [17]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='mother_job_category', y='G3', data=df)
plt.title("Final Grade Distribution by Mother's Job")
plt.xlabel("Mother's Job")
plt.ylabel('Final Grade (G3)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('grade_by_mother_job.png')
plt.close()

In [18]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='father_job_category', y='G3', data=df)
plt.title("Final Grade Distribution by Father's Job")
plt.xlabel("Father's Job")
plt.ylabel('Final Grade (G3)')
plt.xticks(rotation=45)
plt.tight_layout()
plt.savefig('grade_by_father_job.png')
plt.close()

In [19]:
plt.figure(figsize=(12, 6))
sns.scatterplot(x='absences', y='G3', data=df, alpha=0.7)
plt.title('Relationship Between Absences and Final Grades')
plt.xlabel('Number of Absences')
plt.ylabel('Final Grade (G3)')
plt.savefig('absences_vs_grades.png')
plt.close()

In [20]:
plt.figure(figsize=(12, 6))
sns.boxplot(x='absence_category', y='G3', data=df)
plt.title('Grade Distribution by Absence Category')
plt.xlabel('Absence Category')
plt.ylabel('Final Grade (G3)')
plt.savefig('grade_by_absence_category.png')
plt.close()

In [21]:
plt.figure(figsize=(10, 6))
sns.boxplot(x='parents_same_field', y='G3', data=df)
plt.title('Grade Distribution by Whether Parents Work in Same Field')
plt.xlabel('Parents in Same Field (1=Yes, 0=No)')
plt.ylabel('Final Grade (G3)')
plt.xticks([0, 1], ['Different Fields', 'Same Field'])
plt.savefig('grade_by_parent_job_match.png')
plt.close()

In [22]:
plt.figure(figsize=(12, 10))
# Select only numeric columns
numeric_df = df.select_dtypes(include=[np.number])
correlation = numeric_df.corr()
mask = np.triu(correlation)
sns.heatmap(correlation, annot=True, cmap='coolwarm', fmt='.2f', mask=mask)
plt.title('Correlation Heatmap of Numeric Features')
plt.tight_layout()
plt.savefig('correlation_heatmap.png')
plt.close()

In [23]:
plt.figure(figsize=(10, 6))
performance_by_school = pd.crosstab(df['school'], df['performance_category'], normalize='index')
performance_by_school.plot(kind='bar', stacked=True, colormap='viridis')
plt.title('Performance Categories by School')
plt.xlabel('School')
plt.ylabel('Proportion')
plt.legend(title='Performance')
plt.tight_layout()
plt.savefig('performance_by_school.png')
plt.close()

<Figure size 1000x600 with 0 Axes>

In [24]:
plt.figure(figsize=(10, 6))
performance_by_address = pd.crosstab(df['address'], df['performance_category'], normalize='index')
performance_by_address.plot(kind='bar', stacked=True, colormap='viridis')
plt.title('Performance Categories by Address Type')
plt.xlabel('Address Type (U=Urban, R=Rural)')
plt.ylabel('Proportion')
plt.legend(title='Performance')
plt.tight_layout()
plt.savefig('performance_by_address.png')
plt.close()

<Figure size 1000x600 with 0 Axes>

In [25]:
print("\n--- Key Statistical Insights ---")

# Mean grades by school
print("\nMean Grades by School:")
print(df.groupby('school')['G3'].mean())

# Mean grades by address type
print("\nMean Grades by Address Type:")
print(df.groupby('address')['G3'].mean())

# Mean grades by mother's job
print("\nMean Grades by Mother's Job:")
print(df.groupby('mother_job_category')['G3'].mean().sort_values(ascending=False))

# Mean grades by father's job
print("\nMean Grades by Father's Job:")
print(df.groupby('father_job_category')['G3'].mean().sort_values(ascending=False))


--- Key Statistical Insights ---

Mean Grades by School:
school
GP    10.489971
MS     9.847826
Name: G3, dtype: float64

Mean Grades by Address Type:
address
R     9.511364
U    10.674267
Name: G3, dtype: float64

Mean Grades by Mother's Job:
mother_job_category
Healthcare    12.147059
Education     11.051724
Services      11.019417
Other          9.822695
Home           9.152542
Name: G3, dtype: float64

Mean Grades by Father's Job:
father_job_category
Education     11.965517
Healthcare    11.611111
Services      10.297297
Other         10.193548
Home          10.150000
Name: G3, dtype: float64


In [26]:
print("\nAbsence Statistics by Performance Category:")
print(df.groupby('performance_category')['absences'].agg(['mean', 'median', 'min', 'max']))

print("\nParent Job Combinations with Highest Grades:")
job_combo = df.groupby(['mother_job_category', 'father_job_category'])['G3'].mean().sort_values(ascending=False)
print(job_combo.head(5))

print("\nParent Job Combinations with Lowest Grades:")
print(job_combo.tail(5))


Absence Statistics by Performance Category:
                          mean  median  min  max
performance_category                            
Poor                  4.057143     0.0    0   26
Average               7.015464     4.0    0   75
Good                  4.813187     4.0    0   23
Excellent             4.300000     2.0    0   24

Parent Job Combinations with Highest Grades:
mother_job_category  father_job_category
Healthcare           Healthcare             13.500000
Services             Education              13.125000
Education            Education              13.083333
Healthcare           Services               12.400000
Home                 Home                   12.285714
Name: G3, dtype: float64

Parent Job Combinations with Lowest Grades:
mother_job_category  father_job_category
Other                Home                   9.200000
Home                 Other                  8.878788
                     Services               8.800000
Services             Home         

<ipython-input-26-8aaed002ecba>:2: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  print(df.groupby('performance_category')['absences'].agg(['mean', 'median', 'min', 'max']))


In [27]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.preprocessing import OneHotEncoder

X = df[['is_urban', 'absences', 'parents_same_field']]
X = pd.concat([X, df_mother_job, df_father_job], axis=1)
y = df['G3']

model = RandomForestRegressor(n_estimators=100, random_state=42)
model.fit(X, y)

feature_importance = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
}).sort_values('Importance', ascending=False)

print("\nFeature Importance for Predicting Grades:")
print(feature_importance.head(10))


Feature Importance for Predicting Grades:
               Feature  Importance
1             absences    0.364565
0             is_urban    0.098053
2   parents_same_field    0.069032
5          mother_Home    0.064379
11        father_Other    0.057561
8     father_Education    0.057032
7      mother_Services    0.056094
12     father_Services    0.046418
10         father_Home    0.044529
6         mother_Other    0.041122


In [28]:
plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Feature', data=feature_importance.head(10))
plt.title('Top 10 Features for Predicting Grades')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()

engineered_df = df[['school', 'address', 'absences', 'Mjob', 'Fjob', 'G3',
                   'is_urban', 'parents_same_field', 'absence_category',
                   'performance_category', 'mother_job_category', 'father_job_category'] +
                  list(df_mother_job.columns) + list(df_father_job.columns)]

engineered_df.to_csv('students_engineered.csv', index=False)

print("\nFeature engineering complete. Enhanced dataset saved to 'students_engineered.csv'")


Feature engineering complete. Enhanced dataset saved to 'students_engineered.csv'
